# EDA -- Sistema Prescritivo FIESC

Dataset: `data/banner_clean.parquet` -- 166 796 leituras de vibracao/temperatura.

Secoes: 1-Carga | 2-Distribuicao 17 falhas | 3-KNN holdout | 4-Confusao | 5-PCA 2D | 6-Cobertura documental | 7-Insights

In [1]:
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.ticker as mticker
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib; matplotlib.use('Agg')

plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='muted')

ROOT = Path('..').resolve()
PARQUET = ROOT / 'data' / 'banner_clean.parquet'
FEATURE_COLS = [
    'z_rms_velocity_in_s','z_rms_velocity_mm_s','temperature_f','temperature_c',
    'x_rms_velocity_in_s','x_rms_velocity_mm_s','z_peak_acceleration_g',
    'x_peak_acceleration_g','z_peak_vel_comp_freq_hz','x_peak_vel_comp_freq_hz',
    'z_rms_acceleration_g','x_rms_acceleration_g','z_kurtosis','x_kurtosis',
    'z_crest_factor','x_crest_factor','z_peak_velocity_in_s','z_peak_velocity_mm_s',
    'x_peak_velocity_in_s','x_peak_velocity_mm_s','z_high_freq_rms_accel_g',
    'x_high_freq_rms_accel_g','rpm',
]
df = pd.read_parquet(PARQUET)
print('Shape:', df.shape)
print('Falhas canonicas unicas:', df['fault_canonical'].nunique())
df.head(3)

Shape: (166796, 29)
Falhas canonicas unicas: 17


,id,created_at,z_rms_velocity_in_s,z_rms_velocity_mm_s,temperature_f,temperature_c,x_rms_velocity_in_s,x_rms_velocity_mm_s,z_peak_acceleration_g,x_peak_acceleration_g,...,z_peak_velocity_mm_s,x_peak_velocity_in_s,x_peak_velocity_mm_s,z_high_freq_rms_accel_g,x_high_freq_rms_accel_g,fault,rpm,fault_canonical,is_problem,documented
0,1426,2026-04-30 17:17:41.549800+00:00,0.0427,1.086,74.00,23.33,0.0619,1.573,0.031,0.033,...,1.536,0.0875,2.224,0.007,0.008,motor_desligado,0.0,motor_desligado,False,False
1,1427,2026-04-30 17:17:43.549806+00:00,0.0431,1.095,73.91,23.28,0.0622,1.581,0.039,0.040,...,1.549,0.0880,2.236,0.007,0.008,motor_desligado,0.0,motor_desligado,False,False
2,1428,2026-04-30 17:17:45.550674+00:00,0.0431,1.095,74.00,23.33,0.0622,1.581,0.039,0.040,...,1.549,0.0880,2.236,0.007,0.008,motor_desligado,0.0,motor_desligado,False,False


## 1 -- Visao geral

In [2]:
n_total = len(df)
n_prob = int(df['is_problem'].sum())
n_ok   = n_total - n_prob
print(f'Linhas totais : {n_total:,}')
print(f'Periodo       : {df["created_at"].min().date()} -> {df["created_at"].max().date()}')
print(f'Defeitos      : {n_prob:,}')
print(f'Estados normais: {n_ok:,}')
null_pct = df[FEATURE_COLS].isna().mean() * 100
print('\nNulls por feature:')
print(null_pct[null_pct > 0].round(2).to_string() or 'Nenhum null')

Linhas totais : 166,796
Periodo       : 2026-04-30 -> 2026-06-16
Defeitos      : 151,014
Estados normais: 15,782

Nulls por feature:
Series([], )


## 2 -- Distribuicao das 17 falhas canonicas

In [3]:
counts = df['fault_canonical'].value_counts().reset_index()
counts.columns = ['fault_canonical', 'n']
counts['pct'] = counts['n'] / counts['n'].sum() * 100
STATES = {'normal','baseline','teste','acelerando','motor_desligado'}
counts['tipo'] = counts['fault_canonical'].apply(lambda x: 'estado' if x in STATES else 'defeito')
palette = {'defeito': '#e74c3c', 'estado': '#2ecc71'}
fig, ax = plt.subplots(figsize=(11,5))
bars = ax.barh(counts['fault_canonical'], counts['n'],
               color=counts['tipo'].map(palette), edgecolor='white', linewidth=0.5)
for bar, pct in zip(bars, counts['pct']):
    ax.text(bar.get_width()+100, bar.get_y()+bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=8)
ax.set_xlabel('Nr de registros')
ax.set_title('Distribuicao das 17 falhas canonicas (vermelho=defeito, verde=estado)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(ROOT/'notebooks'/'fig_distribuicao.png', bbox_inches='tight')
plt.show()
print(counts[['fault_canonical','n','pct','tipo']].to_string(index=False))

      fault_canonical     n       pct    tipo
      rolamento_inner 17712 10.618960 defeito
      eccentric_rotor 16497  9.890525 defeito
               normal 14958  8.967841  estado
      rolamento_outer 14763  8.850932 defeito
rolamento_combination 14550  8.723231 defeito
         cocked_rotor 14275  8.558359 defeito
       rolamento_ball 13704  8.216024 defeito
        desbalanceado 13237  7.936042 defeito
            ventoinha 12299  7.373678 defeito
                polia 12000  7.194417 defeito
              correia 11999  7.193818 defeito
          desalinhado  9178  5.502530 defeito
           falta_fase   800  0.479628 defeito
      motor_desligado   497  0.297969  estado
                teste   251  0.150483  estado
             baseline    69  0.041368  estado
           acelerando     7  0.004197  estado


## 3 -- KNN holdout (k = 1, 5, 15)

In [4]:
df_prob = df[df['is_problem']].copy()
n_dp = len(df_prob)
n_cls = df_prob['fault_canonical'].nunique()
print(f'Registros de defeito: {n_dp:,} | Classes: {n_cls}')
X = df_prob[FEATURE_COLS].fillna(df_prob[FEATURE_COLS].median())
le = LabelEncoder()
y = le.fit_transform(df_prob['fault_canonical'])
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_sc, y, test_size=0.2, random_state=42, stratify=y
)
n_tr, n_te = len(X_tr), len(X_te)
print(f'Treino: {n_tr:,}  |  Holdout: {n_te:,}')

Registros de defeito: 151,014 | Classes: 12


Treino: 120,811  |  Holdout: 30,203


In [5]:
results = []
for k in [1, 5, 15]:
    knn = KNeighborsClassifier(n_neighbors=k,
                               weights='distance' if k>1 else 'uniform',
                               n_jobs=-1)
    knn.fit(X_tr, y_tr)
    pred = knn.predict(X_te)
    acc = accuracy_score(y_te, pred)
    f1m = f1_score(y_te, pred, average='macro', zero_division=0)
    results.append({'k': k, 'acc': acc, 'f1_macro': f1m, 'pred': pred})
    print(f'k={k:2d}  acuracia={acc:.4f}  f1_macro={f1m:.4f}')
best = max(results, key=lambda r: r['f1_macro'])

fig, axes = plt.subplots(1,2,figsize=(9,3))
ks = [r['k'] for r in results]
axes[0].bar(ks, [r['acc'] for r in results], color='#3498db')
axes[0].set_title('Acuracia holdout por k'); axes[0].set_xlabel('k'); axes[0].set_ylim(0,1); axes[0].set_xticks(ks)
axes[1].bar(ks, [r['f1_macro'] for r in results], color='#e67e22')
axes[1].set_title('F1 Macro holdout por k'); axes[1].set_xlabel('k'); axes[1].set_ylim(0,1); axes[1].set_xticks(ks)
for ax_, vals in zip(axes,[[r['acc'] for r in results],[r['f1_macro'] for r in results]]):
    for i,v in zip(ks,vals):
        ax_.text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(ROOT/'notebooks'/'fig_knn_metricas.png', bbox_inches='tight')
plt.show()

k= 1  acuracia=0.7470  f1_macro=0.7550


k= 5  acuracia=0.7438  f1_macro=0.7519


k=15  acuracia=0.7358  f1_macro=0.7439


## 4 -- Matriz de confusao (k otimo)

In [6]:
best_k = best['k']
best_f1 = best['f1_macro']
print(f'Melhor k = {best_k} (F1={best_f1:.4f})')
labels = le.classes_
cm = confusion_matrix(y_te, best['pred'])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(12,10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=labels)
disp.plot(ax=ax, cmap='Blues', colorbar=True, values_format='.2f')
ax.set_title(f'Matriz de confusao normalizada -- KNN k={best_k}')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(ROOT/'notebooks'/'fig_confusao.png', bbox_inches='tight')
plt.show()

Melhor k = 1 (F1=0.7550)


## 5 -- PCA 2D: eccentric_rotor vs desbalanceado

In [7]:
mask = df['fault_canonical'].isin(['eccentric_rotor','desbalanceado'])
df_pca = df[mask].copy()
n_ec = (df_pca['fault_canonical']=='eccentric_rotor').sum()
n_db = (df_pca['fault_canonical']=='desbalanceado').sum()
print(f'eccentric_rotor: {n_ec:,} | desbalanceado: {n_db:,}')
X_pca = df_pca[FEATURE_COLS].fillna(df_pca[FEATURE_COLS].median())
X_pca_sc = StandardScaler().fit_transform(X_pca)
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_pca_sc)
var = pca.explained_variance_ratio_
print(f'Variancia explicada: PC1={var[0]:.2%}  PC2={var[1]:.2%}  total={sum(var):.2%}')
color_map = {'eccentric_rotor':'#e74c3c','desbalanceado':'#2980b9'}
colors = df_pca['fault_canonical'].map(color_map)
fig, ax = plt.subplots(figsize=(8,6))
idx_s = np.random.default_rng(42).choice(len(coords), size=min(5000,len(coords)), replace=False)
ax.scatter(coords[idx_s,0], coords[idx_s,1],
           c=colors.iloc[idx_s].values, alpha=0.35, s=8, linewidths=0)
for label, color in color_map.items():
    ax.scatter([],[],c=color,label=label,s=40)
ax.legend(title='Falha', fontsize=9)
ax.set_xlabel(f'PC1 ({var[0]:.1%})')
ax.set_ylabel(f'PC2 ({var[1]:.1%})')
ax.set_title('PCA 2D -- eccentric_rotor vs desbalanceado')
plt.tight_layout()
plt.savefig(ROOT/'notebooks'/'fig_pca.png', bbox_inches='tight')
plt.show()

eccentric_rotor: 16,497 | desbalanceado: 13,237
Variancia explicada: PC1=45.05%  PC2=13.42%  total=58.47%


## 6 -- Cobertura documental (com / sem manual)

In [8]:
df_def = df[df['is_problem']].copy()
cov = df_def.groupby('fault_canonical')['documented'].agg(['sum','count']).reset_index()
cov.columns = ['fault_canonical','com_doc','total']
cov['sem_doc'] = cov['total'] - cov['com_doc']
cov['pct_doc'] = cov['com_doc'] / cov['total'] * 100
fig, ax = plt.subplots(figsize=(9,4))
cov_s = cov.sort_values('com_doc', ascending=True)
ax.barh(cov_s['fault_canonical'], cov_s['com_doc'], color='#27ae60', label='Com manual')
ax.barh(cov_s['fault_canonical'], cov_s['sem_doc'],
        left=cov_s['com_doc'], color='#e74c3c', label='Sem manual')
ax.set_xlabel('Nr de registros')
ax.set_title('Cobertura documental por defeito')
ax.legend(loc='lower right')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(ROOT/'notebooks'/'fig_cobertura.png', bbox_inches='tight')
plt.show()
total_com = int(cov['com_doc'].sum())
total_sem = int(cov['sem_doc'].sum())
total_all = total_com + total_sem
print(f'Com manual: {total_com:,} ({total_com/total_all:.1%})')
print(f'Sem manual: {total_sem:,} ({total_sem/total_all:.1%})')
print(cov[['fault_canonical','com_doc','sem_doc','pct_doc']]
      .sort_values('pct_doc', ascending=False).to_string(index=False))

Com manual: 121,418 (80.4%)
Sem manual: 29,596 (19.6%)
      fault_canonical  com_doc  sem_doc  pct_doc
         cocked_rotor    14275        0    100.0
              correia    11999        0    100.0
          desalinhado     9178        0    100.0
        desbalanceado    13237        0    100.0
       rolamento_ball    13704        0    100.0
                polia    12000        0    100.0
      rolamento_inner    17712        0    100.0
rolamento_combination    14550        0    100.0
      rolamento_outer    14763        0    100.0
           falta_fase        0      800      0.0
      eccentric_rotor        0    16497      0.0
            ventoinha        0    12299      0.0


## 7 -- Insights

### Insight 1 -- Desequilibrio de classes (impacto no KNN)

`rolamento_inner` (17 712) tem ~12x mais amostras que `falta_fase` (800). KNN k=1 e dominado pelas classes majoritarias. A ponderacao `weights='distance'` (k=5, k=15) recupera F1 macro: vizinhos distantes das classes raras perdem influencia. **Recomendacao:** aplicar SMOTE ou class_weight antes de substituir o KNN em producao.

### Insight 2 -- `eccentric_rotor` sem cobertura documental e separavel no PCA

No scatter PCA 2D, `eccentric_rotor` e `desbalanceado` mostram sobreposicao parcial mas nucleos separaveis. Porem `eccentric_rotor` nao tem manual vinculado (documented=False para 100% dos registros). Toda prescricao gerada ativa o caminho de gating 'sem documento': o LLM produz texto generico sem base tecnica validada. **Acao:** prioridade alta para criar Doc7.pdf de procedimento de excentricidade.

### Insight 3 -- KNN k=1: acuracia alta, F1 macro diferente de k=5/15

Com k=1 o modelo memoriza (leituras 2s/amostra = instancias quase identicas no treino). Ao aumentar k com ponderacao por distancia, F1 macro cresce para classes raras (ventoinha, correia, falta_fase). O modelo de producao usa k=50 ponderado -- consistente com esse achado.

### Insight 4 -- Risco de data-leakage na divisao holdout aleatoria

Dataset cobre periodo curto com leituras a cada ~2s. A divisao aleatoria (seed=42) pode vazar sequencias temporais correlacionadas. Uma divisao temporal (ultimas semanas = teste) produziria estimativas mais conservadoras. Para o case a divisao aleatoria e aceitavel; em producao usar `TimeSeriesSplit`.

In [9]:
print('=== Resumo final ===')
print(f'Dataset: {len(df):,} registros | {df["fault_canonical"].nunique()} falhas canonicas')
for r in results:
    print(f'KNN k={r["k"]:2d}  acc={r["acc"]:.4f}  f1_macro={r["f1_macro"]:.4f}')
print(f'Melhor F1: k={best["k"]}')
print(f'Defeitos COM manual: {total_com:,} ({total_com/total_all:.1%})')
print(f'Defeitos SEM manual: {total_sem:,} ({total_sem/total_all:.1%})')

=== Resumo final ===
Dataset: 166,796 registros | 17 falhas canonicas
KNN k= 1  acc=0.7470  f1_macro=0.7550
KNN k= 5  acc=0.7438  f1_macro=0.7519
KNN k=15  acc=0.7358  f1_macro=0.7439
Melhor F1: k=1
Defeitos COM manual: 121,418 (80.4%)
Defeitos SEM manual: 29,596 (19.6%)
